# Machine Learning for Membrane Development

A worked example of machine learning (ML) on a table of **membrane** experiments: how a thin film was made, how well water and salt went through it, and how we might *suggest* a better recipe.

**Machine learning** here means a computer finds patterns in past experiments so it can guess the result of a *new* recipe. It is not a physics simulator and it does not “understand” chemistry.

| You need | You do not need |
|----------|-----------------|
| Basic Python (variables, functions, run cells **top to bottom**) | A membrane course, or Bayesian optimization |

The review article uses a **nine-step closed loop**. This notebook **runs Steps 4–7** (tabular FE, EDA, ExtraTrees, SHAP, Ax). Steps **1–3** and **8–9** are short discussions so the numbering matches the paper; they are not automated wet-lab code.

| Paper step | In this notebook |
|------------|------------------|
| 1 Objectives | Discussion + YAML `targets` / search limits |
| 2 Data schema | Load the recipe–KPI table |
| 3 Inspect | Missing values, duplicates |
| 4 FE + EDA | Ion lookups, distributions, heatmap |
| 5 Surrogate | ExtraTrees, train/test, parity |
| 6 xAI (SHAP) | Which inputs the model used |
| 7 Inverse design | Ax search (Pareto in this tutorial) |
| 8 Validate | Discussion: fabricate and measure |
| 9 Refine | Discussion: append rows and retrain |

## Learning objectives

1. Tell **features** (inputs we know) from **targets** (numbers we want to predict).
2. Load a table and spot missing cells and duplicate rows.
3. Add extra chemistry columns (ion size, charge) *before* training.
4. Split data into **train** vs **test**, fit a tree model, and read a **parity plot**.
5. Read **SHAP** plots as *hypotheses*, not as proof of a mechanism.
6. Run a short **inverse design** search (Ax) for **continuous** targets only.

### Two kinds of prediction

- **Regression:** the target is a number (permeability, rejection). This tutorial’s KPIs are regression.
- **Classification:** the target is a label (`pass` / `fail`). The examples include that; Ax search is **not** used for labels.

A high test score means “the model matched held-out rows,” **not** “we discovered the true physics.”

On **Google Colab**, run the bootstrap cell below **before** Setup.


### Prerequisites and setup (Google Colab)

Opening this `.ipynb` from GitHub does **not** download `config.yaml`, `src/`, or `data/`.

> **Next cell:** clone/update the repo and `pip install -e ".[all]"` (core + SHAP + Ax). First time can take several minutes. No Google Drive.

**Repository (repo):** the folder that contains `config.yaml`, `src/`, and `data/`.  
**`-e`:** editable install — imports come from `src/` on disk.

### How to read the output

- `Repo root: /content/ML-MEM-Review`
- `Colab bootstrap done.`

---

**Watch out.** Skipping this cell → missing `config.yaml` or `No module named utl_...`. After a GitHub push, run bootstrap again.  
**Local:** from the repo root, once, in *this* kernel: `pip install -e ".[all]"`. Notes: [`docs/COLAB.md`](../../docs/COLAB.md).


In [ ]:
# @title Bootstrap (clone + install)
# === COLAB BOOTSTRAP (safe locally) =========================================
# Clones Oikiy/ML-MEM-Review if you only opened the .ipynb from GitHub,
# installs the package editable, and cd's to the repo root.
import os
import sys
import subprocess
from pathlib import Path

def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

def _run(cmd, **kwargs):
    print("+", " ".join(cmd) if isinstance(cmd, (list, tuple)) else cmd)
    subprocess.check_call(cmd, **kwargs)

def _find_repo_root(start: Path) -> Path | None:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "config.yaml").is_file() and (candidate / "src").is_dir():
            return candidate
    return None

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

IN_COLAB = _in_colab()
REPO_URL = os.environ.get("ML_MEM_REPO_URL", "https://github.com/Oikiy/ML-MEM-Review.git")
REPO_NAME = os.environ.get("ML_MEM_REPO_NAME", "ML-MEM-Review")
REPO_REF = os.environ.get("ML_MEM_REPO_REF", "main")
COLAB_ROOT = Path(os.environ.get("ML_MEM_COLAB_ROOT", f"/content/{REPO_NAME}"))
EXTRAS = 'all'

def _git_head(repo: Path) -> str:
    try:
        return subprocess.check_output(
            ["git", "-C", str(repo), "rev-parse", "HEAD"], text=True
        ).strip()
    except (subprocess.CalledProcessError, FileNotFoundError, OSError):
        return ""

def _truthy(name: str) -> bool:
    return os.environ.get(name, "").strip().lower() in {"1", "true", "yes"}

if IN_COLAB:
    if COLAB_ROOT.exists() and (COLAB_ROOT / ".git").is_dir():
        if _truthy("ML_MEM_SKIP_PULL"):
            print(f"Skipping git update at {COLAB_ROOT} (ML_MEM_SKIP_PULL=1)")
        else:
            print(f"Updating clone at {COLAB_ROOT} to origin/{REPO_REF}")
            _run(["git", "-C", str(COLAB_ROOT), "fetch", "--depth", "1", "origin", REPO_REF])
            _run(["git", "-C", str(COLAB_ROOT), "reset", "--hard", "FETCH_HEAD"])
    elif not COLAB_ROOT.exists():
        _run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(COLAB_ROOT)])
    os.chdir(COLAB_ROOT)
    root = _find_repo_root(Path.cwd())
else:
    root = _find_repo_root(Path.cwd())

if root is None:
    raise RuntimeError(
        "Could not locate the ML-MEM-Review repo (need config.yaml + src/). "
        "On Colab: re-run this cell or set ML_MEM_REPO_URL to your fork. "
        "Locally: open the notebook from inside a full clone of the repository."
    )

os.chdir(root)
print(f"Repo root: {root}  git={_git_head(root)[:12]}")

src = root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

if IN_COLAB:
    marker = root / ".colab_install_ok"
    force = _truthy("ML_MEM_FORCE_INSTALL")
    head = _git_head(root)
    extras = EXTRAS
    pip_spec = str(root) if not extras else f"{root}[{extras}]"
    install_key = f"{head}\n{extras}"
    prev = marker.read_text(encoding="utf-8").strip() if marker.is_file() else ""
    if force or prev != install_key:
        _run([sys.executable, "-m", "pip", "install", "-q", "-e", pip_spec])
        try:
            marker.write_text(install_key + "\n", encoding="utf-8")
        except OSError:
            pass
    else:
        print("Skipping pip install (clone SHA + extras unchanged). Set ML_MEM_FORCE_INSTALL=1 to reinstall.")
    print("Colab bootstrap done.")
else:
    print("Local session: repo root + src/ on sys.path (no auto pip install).")

if EXTRAS:
    try:
        import ax  # noqa: F401
    except ImportError:
        pip_spec = f"{root}[{EXTRAS}]"
        print(f"Ax not found in this kernel; installing {pip_spec}")
        _run([sys.executable, "-m", "pip", "install", "-e", pip_spec])

print("Next: run the Setup import cell below.")
# === end bootstrap ===========================================================


## Setup (before the nine steps)

Load **`config.yaml`** — the control panel for this notebook — and set a random seed so splits are repeatable.

> **Next cell:** print dataset / lookup / `models/` / `outputs/figures/` paths. On Colab they sit under `/content/ML-MEM-Review/`.

### Why it matters

Later cells **read this file**; they should not hard-code the Excel name. To reuse the package on *your* table, edit YAML, not this Python. Same engine as:

```bash
python -m run_pipeline --config config.yaml
```

### New words

| Term | Meaning |
|------|---------|
| Random seed | Integer that makes “random” shuffles repeatable (`modeling.random_state`). |
| Kernel | The Python process behind the notebook. Installs must go here. |

### YAML

```yaml
data:
  dataset: "TutorialDataSet.xlsx"   # file inside data/raw/
  targets:                          # columns to predict
    - "Pure_Water_Permeability_..."
    - "solute1_Rejection"
```

---

**Watch out.** Run bootstrap before this cell on Colab. An Ax-missing warning is OK until Step 7.  
**Try.** Does the Excel path come from Python you typed, or from `config.yaml`?


In [ ]:
import os
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from ax.api.client import Client
    from ax.api.configs import RangeParameterConfig, ChoiceParameterConfig
except Exception as exc:
    Client = None
    RangeParameterConfig = None
    ChoiceParameterConfig = None
    warnings.warn(
        f"Ax import skipped during setup: {exc}. "
        "You can still run Steps 2-6; install pip install -e '.[opt]' before Step 7.",
        RuntimeWarning,
    )

PROJECT_ROOT = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(PROJECT_ROOT, "config.yaml")):
    parent = os.path.dirname(PROJECT_ROOT)
    if parent == PROJECT_ROOT:
        raise RuntimeError(
            "Could not find config.yaml above the current folder. "
            "Launch the notebook from inside the repository."
        )
    PROJECT_ROOT = parent
CODE_PATH = os.path.join(PROJECT_ROOT, "src")
if CODE_PATH not in sys.path:
    sys.path.insert(0, CODE_PATH)

from utl_config_loader import (
    get_data_paths,
    get_figures_dir,
    get_model_dir,
    load_config,
    resolve_targets,
)

config = load_config(project_root=PROJECT_ROOT)
print("Configuration loaded from:", config["_config_path"])
print("Same engine as: python -m run_pipeline --config config.yaml")

plotting_cfg = config["plotting"]
FONT_SIZE = plotting_cfg["font_size"]
FIG_SIZE = tuple(plotting_cfg["global_figure_size"])
DPI = plotting_cfg["dpi"]
plt.rcParams["font.size"] = FONT_SIZE
plt.rcParams["figure.figsize"] = FIG_SIZE
plt.rcParams["savefig.dpi"] = DPI

RANDOM_STATE = config["modeling"]["random_state"]
np.random.seed(RANDOM_STATE)

data_paths = get_data_paths(config)
DATA_PATH = data_paths["dataset"]
lookup_by_align_column = data_paths.get("lookup_by_align_column", {})
CATION_PROP_PATH = lookup_by_align_column.get("S1_Cation")
ANION_PROP_PATH = lookup_by_align_column.get("S1_Anion")
if CATION_PROP_PATH is None or ANION_PROP_PATH is None:
    raise KeyError(
        "config.yaml must define feature_engineering.lookups for S1_Cation and S1_Anion."
    )

MODEL_PATH = get_model_dir(config)
FIGURES_DIR = get_figures_dir(config)
os.makedirs(FIGURES_DIR, exist_ok=True)
targets = resolve_targets(config)

print("Dataset:", os.path.basename(DATA_PATH))
print("Targets:", targets)
print("Cation lookup:", os.path.basename(CATION_PROP_PATH))
print("Anion lookup:", os.path.basename(ANION_PROP_PATH))
print("Models:", MODEL_PATH)
print("Figures:", FIGURES_DIR)


## Step 1. Objectives (discussion)

The paper’s loop starts by writing down **what “good” means** *before* fitting a model: which **KPIs** (e.g. water permeability, salt rejection), which way to push them (usually maximize both), and **lab limits** (what you can actually mix, how long you can polymerize, which solvents you will not change).

This repository does **not** invent those choices. You record them in `config.yaml`:

```yaml
data:
  targets:
    - "Pure_Water_Permeability_{Lm^{-2}^h{-1}bar^{-1}}"
    - "solute1_Rejection"
optimization:
  mode: multiobjective          # both KPIs at once (Pareto)
  search_space: ...             # knobs you are willing to change
  constant_parameters: ...      # knobs you refuse to change
```

**In the paper:** Step 1 is a design meeting. **In this notebook:** the YAML *is* that meeting, already filled for the tutorial membrane. Later steps only read it.

No extra code cell — Setup already loaded this file.


## Step 2. Data schema — load the table

Read the spreadsheet into a **DataFrame** (`df`: a table with named columns) and inspect size and names.

> **Next cell:** `df.shape` and `df.head()`.

### Why it matters

**One row = one published experiment**, not a random sample from nature. Duplicate recipes can make scores look better than they are.

### New words

| Role | Meaning | Examples |
|------|---------|----------|
| **Feature** (input) | What we know before predicting | monomer name, mass, time, `S1_Cation` |
| **Target** (output) | The number we want to predict | water permeability, solute rejection |
| **KPI** | An important target | how fast water goes through; how well salt is rejected |

The bundled table is ~**278 rows × 36 columns**. Another file in YAML → different numbers. Long names with `{Lm^{-2}...}` are **units in the column name**.

### How to read the output

Look for `DOI`, `Dope1_Monomer1_Name`, `solute1_Rejection`.

---

**Try.** Write down **one feature** and **one target** from the column list before continuing.


In [ ]:
# Load the dataset into a pandas DataFrame
df = pd.read_excel(DATA_PATH)

# Show the shape and first few rows of the dataset
print("Dataset shape (rows, columns):", df.shape)
df.head()

### How the columns are grouped

1. **How the film was made** — monomers, additives, solvents, time (`Dope1_...`, `Dope2_...`).
2. **How it was tested** — pressure, feed, compaction *when reported*.
3. **What was in the water** — `Solute1_Name`, `S1_Cation`, `S1_Anion`.

Step 4a attaches **numbers** (size, charge) to those ion *names*. Still looking, not cleaning.


## Step 3. Inspect — missing values and duplicates

Count empty cells and exact duplicate rows.

> **Next cell:** `df.info()` and a duplicate count.

### New words

| Term | Meaning |
|------|---------|
| Missing value | Blank in the sheet; pandas shows `NaN` (“not a number”). |
| Duplicate row | The same experiment copied twice — can inflate scores. |
| Imputation | Filling blanks. Here it happens **later**, **inside training**, on the **train set only**. |
| Leakage | Using test-set information while learning. |

We do **not** fill blanks in this cell.

---

**Watch out.** Dropping every row with *any* blank can wipe a small materials table.  
**Try.** If polymerization time is missing in 80% of rows: impute, drop the column, or keep an “is missing?” flag?


In [ ]:
# Check the data types and missing values
print("\nData types and missing values:")
print("---"*30)
print(df.info())

# Check for duplicates
num_duplicates = df.duplicated().sum()
print(f"\nNumber of duplicate rows: {num_duplicates}")

### Optional deeper dive: why values are missing

Skip if you want. No imputation in the next code cell.

| Pattern | Meaning |
|---------|---------|
| **MCAR** | Missing by accident. Simple averages often OK if few blanks. |
| **MAR** | Missingness depends on **other recorded** columns. |
| **MNAR** | Missing *because of* the missing value (extremes never published). |
| **Structured** | Never in the protocol — dropping the column is more honest than a fake mean. |

Step 5 fills holes using **training data only**, then applies the same rule to test rows.


In [ ]:
# Implement duplicate-row handling identified above.
num_duplicates = df.duplicated().sum()
if num_duplicates > 0:
    print(f"Dropping {num_duplicates} duplicate rows (keeping the first occurrence).")
    df = df.drop_duplicates(keep="first").reset_index(drop=True)
    print(f"Rows remaining after deduplication: {len(df)}")
else:
    print("No duplicate rows to drop.")


In [ ]:
from utl_config_loader import resolve_features_to_model, resolve_targets

targets = resolve_targets(config)
print("data.targets in config.yaml:")
for name in targets:
    print(" -", name)
    if name not in df.columns:
        raise ValueError(f"Target {name!r} is not a column in the loaded table.")

print("\nfeatures.include lists model inputs. Ion-size columns (S1_cat_*, S1_an_*)")
print("are created in Step 4a; they are not on the raw table yet.")
print("Configured feature names:")
for name in config.get("features", {}).get("include") or []:
    print(" -", name)


## Step 4. Feature engineering and EDA

### 4a. Feature engineering

Create extra input columns *before* fitting, by **joining** ion property tables onto `S1_Cation` / `S1_Anion`.

> **Next cell:** `feature_engineering(...)` → `df_enriched`. Printed feature list should include `S1_cat_*` / `S1_an_*`.

### Why it matters

A name like `Na+` is only text. Transport cares about **size** and **charge**. The lookup files are small dictionaries: ion name → numbers. This is **not** training and **not** a simulator.

### YAML

```yaml
feature_engineering:
  lookups:
    - align_column: "S1_Cation"    # column in the main table
      lookup_key: "Cation"         # column in the ion file
      lookup_path: "data/raw/cation_prop.xlsx"
```

Ion columns are listed in `features.include` but do **not** exist on the raw table.

---

**Watch out.** Ion names that do not match the lookup stay without those extra columns.  
**Try.** After the cell, find a column that was **not** in Step 2’s `df.head()`.


In [ ]:
from utl_feateng import feature_engineering
from utl_config_loader import resolve_features_to_model

df_enriched = feature_engineering(df, config=config, verbose=True)
features = resolve_features_to_model(config, df_enriched)
print(f"\n{len(features)} model features after enrichment (from config.yaml):")
for name in features:
    print(" -", name)
df_enriched


### The enriched table

`df_enriched` = original recipe columns **plus** ion descriptors. This tutorial only adds a small ion set; you could add solvent or polymer properties the same way in YAML.

For solutes you care about, `S1_Cation` / `S1_Anion` should become real numbers, not blanks.


### Optional: columns that never change

**Variance** = how much a column wiggles. A constant temperature cannot help a prediction, but after **one-hot encoding** (names → 0/1 columns, Step 5) it can add clutter.

> **Next cell:** *report* near-constant columns. Modeling still uses `features.include` (same as `run_pipeline`).

Empty printed list = nothing was constant enough to flag.


In [ ]:
from utl_feateng import remove_low_variance_columns

variance_threshold = config.get("features", {}).get("low_variance_threshold", 1e-9)
_, dropped = remove_low_variance_columns(
    df_enriched,
    exclude_columns=targets,
    variance_threshold=variance_threshold,
)
print("Variance threshold:", variance_threshold)
print("Near-constant columns (not dropped for modeling; train_model uses config features):", dropped)


### Which knobs we keep

`features.include` is the allow-list of **model inputs** after enrichment. Step 7 **searches** only a subset (lab-tunable knobs). Everything else the model needs is held **constant** during search.

Plots and training both use `df_enriched` and the same YAML as `python -m run_pipeline`.


### 4b. Exploratory plots (before training)

Look at how the targets are spread and which numbers move together — quality control, not the conclusion.

> **Next cell:** `target_distributions.png` and `pearson_heatmap.png` under `outputs/figures/`.

### Why it matters

If permeability sits in a tiny band, a “good” score can be cheap (always guess the average). If two numeric columns are copies, the model may pick either.

### New words

| Term | Meaning |
|------|---------|
| Pearson correlation | −1 to 1 for a *straight line* between two **numeric** columns. Ignores monomer *names*. |

Near 1: they rise together; near −1: opposite; near 0: no linear link. The heatmap hides pairs with \|r\| below the YAML threshold (here **0.5**).

### How to read the output

- **Distribution:** spread, skew, outliers (unusual rows, not automatically errors).
- **Heatmap:** strong r ≠ causation in a literature table.

---

**Watch out.** Do not drop a lab knob only because Pearson is small — trees can still use nonlinear effects.  
**Try.** Will monomer *name* appear on the heatmap? (No.)


In [ ]:
from utl_eda import plot_target_distributions, compute_and_plot_pearson_heatmap
from utl_fileIO import file_io

fig, _ = plot_target_distributions(df_enriched, target_columns=targets, config=config)
file_io("save", "figure", FIGURES_DIR, "target_distributions.png", data=fig)
print("Saved target_distributions.png")

numeric_df = df_enriched.select_dtypes(include=["number"])
threshold = (config.get("data_analysis", {}) or {}).get("correlation_thresholds", [0.5])[-1]
_, heatmap_fig = compute_and_plot_pearson_heatmap(
    numeric_df, threshold=threshold, config=config
)
file_io("save", "figure", FIGURES_DIR, "pearson_heatmap.png", data=heatmap_fig)
print("Saved pearson_heatmap.png")


### Reading the heatmap

Strong r can mean “papers report these two settings together,” not “A causes B.” Weak Pearson ≠ “useless.” Step 5 uses **trees** so *combinations* (monomer × time × additive) can still matter.

If two numerics are almost the same measurement, keep the one you can set in the lab — that also matters for the Step 7 search space.


## Step 5. Surrogate — ExtraTrees

For each `data.targets` column, fit a model and draw a **parity plot** (predicted vs measured).

> **Next cell:** train/test metrics, `parity_<target>.png`, pickles in `models/`. `force_retrain=True` so YAML changes are picked up.

### Why it matters

We split rows at random:

- **Train** (~80%): the model may learn from these.
- **Test** (~20%): held back. If it also does well here, it is not only memorizing homework.

Train great + test poor = **overfitting** (answer key vs new exam).

### New words

| Term | Meaning |
|------|---------|
| ExtraTrees | Crowd of **decision trees** (yes/no questions like “time > 5 min?”). Average many random trees. Treat as a flexible table fitter. |
| One-hot encoding | Category names → 0/1 columns (PIP=1, PEI=0, …), fit on **train only**. |
| R² | Often 0–1 (can be negative). 1 = perfect; 0 = no better than predicting the average. Compare **train vs test**. |
| RMSE / MAE | Typical error in the **same units as the KPI**. Smaller is better. RMSE punishes big misses more. |

### YAML

```yaml
modeling:
  model_selection: extratrees   # or randomforest, ridge, ...
  task: regression
  train_test_ratio: 0.2
```

Change `model_selection` in YAML, not in this cell.

### How to read the output

Points near the **diagonal** = good. A long target name in the title is normal.

---

**Watch out.** Do not trust train R² alone.  
**Try.** Train R² = 0.99 and test R² = 0.2 — would you trust Step 7’s recipes? Why or why not?


In [ ]:
from utl_fileIO import file_io
from utl_modelling import _safe_target, evaluate_and_plot_results, train_model

for target in targets:
    print(f"\n=== Modeling target: {target} ===")
    pipe, X_train, X_test, y_train, y_test = train_model(
        df=df_enriched,
        target=target,
        config_path=config["_config_path"],
        force_retrain=True,
    )
    fig, resolved_task = evaluate_and_plot_results(
        target=target,
        y_train=y_train,
        y_pred_train=pipe.predict(X_train),
        y_test=y_test,
        y_pred_test=pipe.predict(X_test),
        config=config,
        fig_size=FIG_SIZE,
    )
    plot_name = (
        f"confusion_{_safe_target(target)}.png"
        if resolved_task == "classification"
        else f"parity_{_safe_target(target)}.png"
    )
    file_io("save", "figure", FIGURES_DIR, plot_name, data=fig)
    print("Saved", plot_name)


### How to read a parity plot

Each point is one experiment: **measured** vs **predicted**.

- **Dashed diagonal:** prediction = measurement.
- **Above / below** the line: over- / under-predicted.
- Train tight, test scattered → overfitting.
- Bend at high permeability → weak there; inverse-design suggestions in that region are shaky.

Classification uses a **confusion matrix** instead. This tutorial’s KPIs are numbers → parity.

**Try.** Where would “always under-predicts high rejection” show up?


## Step 6. xAI (SHAP)

A test score says *whether* predictions match. **SHAP** says *which input columns* pushed a prediction up or down **in this fitted model**.

> **Next cell:** `shap_summary_*.png` and `shap_importance_*.png`. If `shap` is missing, the cell tries to install `.[shap]` or skips so you can still run Step 7.

### Why it matters

That is an explanation of the **surrogate**, not a proof of membrane physics, and **not causation**. You do not need the game-theory math.

### New words

| Term | Meaning |
|------|---------|
| SHAP | SHapley Additive exPlanations. Top rows matter more on average. Color = high vs low feature value; left/right = down vs up in the prediction. |

Only the **top 20** encoded features are drawn if there are many (one-hot names get long).

---

**Watch out.** Not “SHAP says additive loading *causes* rejection.” Safer: “In this model it was *associated with* …”  
**Try.** Circle one lab-tunable feature. That is a candidate for the Ax **search space**; the rest belong in **constants**.


### Beeswarm vs bar

- **Beeswarm (summary):** each dot is one sample — use for **direction**.
- **Bar (importance):** mean \|SHAP\| — a **ranking** without direction.

Names like `Dope1_Monomer1_Name_PIP` are one-hot columns from Step 5. That is expected.


In [ ]:
try:
    import shap  # noqa: F401
except ImportError:
    import subprocess
    spec = f"{PROJECT_ROOT}[shap]"
    print("SHAP not found; installing", spec)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", spec])
    import shap  # noqa: F401

from utl_config_loader import resolve_features_to_model
from utl_fileIO import file_io
from utl_modelling import _safe_target
from utl_shap import shap_explain_target

features = resolve_features_to_model(config, df_enriched)
for target in targets:
    try:
        shap_values, feature_names, summary_fig, bar_fig = shap_explain_target(
            target=target,
            df_enriched=df_enriched,
            features_to_model=features,
            config=config,
            max_samples=1000,
        )
    except ImportError as exc:
        print(exc)
        print("Skipping SHAP. You can continue with Step 7.")
        break
    if summary_fig is None or bar_fig is None:
        print("SHAP skipped for", target)
        continue
    safe = _safe_target(target)
    file_io("save", "figure", FIGURES_DIR, f"shap_summary_{safe}.png", data=summary_fig)
    file_io("save", "figure", FIGURES_DIR, f"shap_importance_{safe}.png", data=bar_fig)


### From plots to a hypothesis

If monomer, additive, time, or ion size rank high *and* the direction matches interfacial polymerization, the model is at least consistent with chemistry. If the top feature is a paper ID or a constant lab column, the feature list is wrong.

Step 7 should only vary a **short** list of actionable knobs. If that list fights SHAP *and* common sense, fix the YAML before trusting Ax.


## Step 7. Inverse design with Ax

Treat ExtraTrees as a cheap stand-in for “make the membrane and measure it,” and let **Ax** search recipes that trade off permeability and rejection.

> **Next cells:** build the search space from YAML, run trials, print recipes, then figures. First Ax install can take several minutes (`pip install -e ".[opt]"`).

### Why it matters

**Inverse design** = we know the performance we want and search for *inputs*. The optimizer **never** makes a real film; it only queries the model.

**Analogy.** Flight simulator (ExtraTrees) vs real aircraft (the lab). Keep a shortlist from the simulator; still fly the plane.

This tutorial uses **`optimization.mode: multiobjective`** (a **Pareto set**, next subsection), not one weighted winner.

### New words

| Term | Meaning |
|------|---------|
| Ax | Open-source **Bayesian optimization** library. |
| Bayesian optimization | Propose the next recipe using a model of “where good results might be,” mixing new regions (**exploration**) with known good ones (**exploitation**). |

### YAML

```yaml
optimization:
  mode: multiobjective
  n_trials: 100          # more trials = slower, often a better front
  search_space: ...      # knobs Ax may change
  constant_parameters:   # knobs held fixed
```

### How to read the output

Completed trials, printed recipes, `inverse_design_summary.png`. On Colab: `/content/ML-MEM-Review/outputs/figures/`.

---

**Watch out.** Not for **classification** labels. Need Step 5 `.pkl` files. After changing `mode`, delete `models/ax_client.pkl` or keep `force_rebuild=True`.  
**Try.** On a Pareto plot, what would high permeability *and* high rejection mean — and why might no trial sit there?


### Search space vs constants

| | Meaning |
|--|---------|
| **Search space** | Knobs Ax may change — ranges (time, mass) or choices (monomer). `optimization.search_space`. |
| **Constants** | Model inputs you **refuse** to vary. `constant_parameters` (e.g. solvent, test salt). |

**Hard rule.** Searchable + constant columns together must match training. Keep the space **small** so suggestions stay lab-plausible.

> **Next cell:** how many Ax parameters were built. Check that list against your protocol.


In [ ]:
if Client is None or RangeParameterConfig is None or ChoiceParameterConfig is None:
    import subprocess
    spec = f"{PROJECT_ROOT}[opt]"
    print("Ax not found; installing", spec)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", spec])
    from ax.api.client import Client
    from ax.api.configs import RangeParameterConfig, ChoiceParameterConfig

from utl_Ax import (
    build_ax_search_space,
    get_optimization_recommendations,
    oracle_from_params,
    plot_ax_results,
    results_to_dataframe,
    run_ax_optimization,
)


In [ ]:
opt_cfg = config["optimization"]
search_space_parameters = build_ax_search_space(
    opt_cfg=opt_cfg,
    choice_param_cls=ChoiceParameterConfig,
    range_param_cls=RangeParameterConfig,
)
print("optimization.mode:", opt_cfg.get("mode"))
print("Search parameters from config.yaml:", len(search_space_parameters))
for item in search_space_parameters:
    print(" -", getattr(item, "name", item))


### The oracle (the model as a fake experiment)

The **oracle** is “ask the saved models.” Each trial, `oracle_from_params`:

1. Merge proposed knobs with `constant_parameters`.
2. One-row table + the **same** `feature_engineering` as Step 4a.
3. Predict each KPI with saved ExtraTrees pipelines.
4. Hand numbers back to Ax.

You do not call it yourself; `run_ax_optimization` does.

---

**Watch out.** Missing `models/*.pkl` (go back to Step 5) or a YAML name that is not a training feature.


### Two goals: a Pareto front, not one score

Permeability and rejection **compete**. There is a **set** of good recipes, not a single winner.

> **Next cell:** prints `mode` and each objective’s direction and threshold.

### New words

| Term | Meaning |
|------|---------|
| Dominated | Another recipe is better on **both** KPIs. |
| Pareto frontier | Recipes that are *not* dominated: you cannot improve one KPI without hurting the other. |
| Threshold (YAML) | Soft constraint on the **normalized** KPI (scaled ~0–1 with `normalization_min` / `max`). |

### How to read the maps

- **Blue triangles** = Pareto. **Red circles** = all trials.

### Optional deeper dive

`mode: scalarized` + `scalarized_objective.equation` collapses two KPIs into **one weighted number** (example 05). Faster, but “best” is only best *for those weights*.


In [ ]:
opt_cfg = config["optimization"]
print("mode:", opt_cfg.get("mode"))
for obj in opt_cfg.get("objectives") or []:
    print(
        f"  - {obj.get('name')}: {obj.get('direction')} "
        f"threshold={obj.get('threshold')} "
        f"-> {obj.get('model_target') or obj.get('original_name')}"
    )
eq = (opt_cfg.get("scalarized_objective") or {}).get("equation")
if opt_cfg.get("mode") == "scalarized":
    print("scalarized_objective.equation:", eq)


### The optimization loop

A few structured-random (**Sobol**) starts, then Bayesian trials. `n_trials` is the budget. Large values (the tutorial file uses 100) can take **many minutes**.

> **Next cell:** `run_ax_optimization`. Checkpoint under `models/` (`client_cache_file`).

**Sobol** = diverse random starts, better than throwing darts blindly. Re-running without `force_rebuild=True` may **reload** the checkpoint instead of searching again.

Treat the result as a **shortlist of experiments**, not a new membrane.

---

**Watch out.** No Ax in this kernel → bootstrap or `pip install -e ".[opt]"`.


In [ ]:
client, results = run_ax_optimization(
    search_space_parameters=search_space_parameters,
    oracle_from_params=oracle_from_params,
    config=config,
    sobol_init=opt_cfg["sobol_init"],
    n_trials=opt_cfg["n_trials"],
    force_rebuild=True,
    random_seed=config.get("modeling", {}).get("random_state"),
    save_every_trials=opt_cfg["save_checkpoint_every"],
    client_path=os.path.join(MODEL_PATH, opt_cfg["client_cache_file"]),
)
print("Optimization completed:", len(results), "trials")


### Reading the suggested recipes

> **Next cell:** `get_optimization_recommendations` — a **Pareto set** (several permeability–rejection trade-offs) under *this* search box and ExtraTrees surrogate.

Not a universally optimal membrane. Recipes should look like something a chemist would mix.

---

**Watch out.** Negative mass or a nonsense monomer pairing → surrogate, bounds, or constants are wrong.  
**Try.** Would you run the top recipe this week? What measurement would make you trust or reject the surrogate?


In [ ]:
recommendations = get_optimization_recommendations(
    client=client,
    optimization_mode=opt_cfg["mode"],
    top_k=5,
)


### Pictures of the search

> **Next cell:** `inverse_design_summary.png` (metrics vs trial — 3D “waterfall”) and **response maps**.

| Figure | How to read it |
|--------|----------------|
| Waterfall | Each ribbon is one quantity vs trial index (0, 1, 2, …). You want movement after the random start, not a flat line. |
| Response map | Smoothed landscape vs two knobs from **these few trials** — a cartoon, not nature. **Contour** = 2D; **surface** = 3D of the **same** interpolation. |
| Symbols | Red circles = all trials. Blue triangles = Pareto (if `mode: multiobjective`). |

If a map is skipped (too few distinct points), the summary figure is still enough. Files: figures directory from Setup.


In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display
import matplotlib.pyplot as plt
from utl_fileIO import file_io

trial_df = results_to_dataframe(results)
viz = plot_ax_results(trial_df=trial_df, config=config, client=client)

summary_figure = (
    viz.get("summary", {}).get("figure") if isinstance(viz.get("summary"), dict) else None
)
if summary_figure is None and viz.get("figures"):
    summary_figure = viz["figures"].get("summary_figure")
if summary_figure is not None:
    file_io("save", "figure", FIGURES_DIR, "inverse_design_summary.png", data=summary_figure)

for index, response_map in enumerate(viz.get("response_maps", []), start=1):
    if response_map.get("contour_figure") is not None:
        file_io(
            "save", "figure", FIGURES_DIR,
            f"response_map_{index}_contour.png",
            data=response_map["contour_figure"],
        )
    if response_map.get("surface_figure") is not None:
        file_io(
            "save", "figure", FIGURES_DIR,
            f"response_map_{index}_surface.png",
            data=response_map["surface_figure"],
        )

plt.close("all")
for name in [
    "inverse_design_summary.png",
    "response_map_1_contour.png",
    "response_map_1_surface.png",
    "response_map_2_contour.png",
    "response_map_2_surface.png",
]:
    path = Path(FIGURES_DIR) / name
    if path.is_file():
        display(Markdown(f"### {name}"))
        display(Image(filename=str(path)))


## Step 8. Validate in the lab (discussion)

Steps 5–7 only queried a **surrogate**. The paper’s next step is wet-lab: **fabricate** one or a few suggested recipes and **measure** the same KPIs (permeability, rejection) under the same test conditions as the table.

This repository does **not** drive a coater, a cell, or a robot. After Ax prints a shortlist:

1. Pick 1–3 chemically plausible Pareto (or scalarized) recipes.
2. Make the films.
3. Measure KPIs with the same protocol as `TutorialDataSet.xlsx`.
4. Write the new rows (successes **and** failures) back into the table.

Failures belong in the schema too — otherwise the next model never learns the dead ends.

There is no code cell here on purpose.


## Step 9. Refine and close the loop (discussion)

The paper’s last step is: **append** the new measurements, **retrain** the surrogate, and run inverse design again. That is the orange feedback arrow into the data schema (Step 2).

In this package that means:

1. Add rows to the Excel/CSV (same columns as Step 2).
2. Re-run Steps 3–7 (`force_retrain=True`, delete or rebuild `models/ax_client.pkl`).
3. Compare the new Pareto set to the last one.

The companion code **stops** at Step 7 so the same YAML workflow can run on any tabular dataset, including those with no lab attached. Steps 8–9 are the experiment, not extra Python.


## Recap, troubleshooting, next steps

**Paper loop:** Steps 1–9. **This code:** Steps 4–7. **Discussed only:** 1–3 and 8–9.

### Key concepts

- **Features** vs **targets**; **train** vs **test**; **correlation** ≠ **causation**.
- The model is a **surrogate**. SHAP explains it. Ax **searches** it.
- Inverse design is for **continuous** KPIs (Step 7). Tutorial YAML = **multiobjective (Pareto)**; example 05 = **scalarized** demo.
- After the shortlist: **Step 8** lab test, **Step 9** add rows and retrain — not automated here.

### Troubleshooting

| Problem | What to try |
|---------|-------------|
| `config.yaml` / `utl_*` not found | Bootstrap, then Setup, in order. |
| `No module named shap` | `pip install -e ".[shap]"` in **this** kernel; or skip to Step 7. |
| Ax / optimize error | `pip install -e ".[opt]"`; confirm Step 5 wrote `.pkl` files. |
| No Pareto triangles | `mode: multiobjective`; delete `models/ax_client.pkl`; `force_rebuild=True`. |
| Old figures / old model | `force_retrain=True` / `force_rebuild=True`, or delete `models/`. |
| Colab clone looks old | Re-run bootstrap after a GitHub push. |

### Practice (pick one)

1. Change `data.targets` to another continuous column; re-run Sections 4–5.
2. Set `modeling.model_selection: randomforest`; compare test RMSE.
3. Set `optimization.mode: scalarized`; compare that winner to the Pareto set.

None of this replaces making the membrane. It helps you choose **which** membrane to make next.
